In [14]:
import os
import json
import math
import random
import time
import argparse
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple

import cv2
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import CLIPModel, CLIPTokenizerFast, get_cosine_schedule_with_warmup

In [15]:
# ============================================================
# BLOCK 1: Utility helpers
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def save_json(obj: Dict[str, Any], path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def format_seconds(seconds: float) -> str:
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"

In [16]:
# ============================================================
# BLOCK 2: Config and argument parsing
# ============================================================

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Fine-tune CLIP on MSVD for text-video retrieval")

    parser.add_argument("--train_json", type=str, required=True)
    parser.add_argument("--val_json", type=str, required=True)
    parser.add_argument("--video_root", type=str, required=True)
    parser.add_argument("--output_dir", type=str, required=True)

    parser.add_argument("--model_name", type=str, default="openai/clip-vit-base-patch32")
    parser.add_argument("--image_size", type=int, default=224)
    parser.add_argument("--max_text_len", type=int, default=32)
    parser.add_argument("--num_frames", type=int, default=8)

    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--num_workers", type=int, default=4)
    parser.add_argument("--epochs", type=int, default=10)
    parser.add_argument("--lr", type=float, default=1e-5)
    parser.add_argument("--weight_decay", type=float, default=1e-4)
    parser.add_argument("--warmup_ratio", type=float, default=0.1)
    parser.add_argument("--grad_accum_steps", type=int, default=2)
    parser.add_argument("--max_grad_norm", type=float, default=1.0)

    parser.add_argument("--freeze_vision", action="store_true")
    parser.add_argument("--freeze_text", action="store_true")
    parser.add_argument("--train_text_projection_only", action="store_true")
    parser.add_argument("--train_vision_projection_only", action="store_true")
    parser.add_argument("--unfreeze_last_n_vision", type=int, default=4)
    parser.add_argument("--unfreeze_last_n_text", type=int, default=4)

    parser.add_argument("--save_every", type=int, default=1)
    parser.add_argument("--resume", type=str, default="")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--amp", action="store_true")
    parser.add_argument("--log_every", type=int, default=50)

    return parser.parse_args()

In [17]:
# ============================================================
# BLOCK 3: MSVD JSON parsing
# ============================================================

def build_pairs_from_msvd_json(json_path: str, video_root: str) -> List[Dict[str, str]]:
    with open(json_path, "r", encoding="utf-8") as f:
        records = json.load(f)

    pairs: List[Dict[str, str]] = []
    missing_videos = 0

    for item in records:
        video_name = item["video"]
        captions = item["caption"]
        video_path = os.path.join(video_root, video_name)

        if not os.path.exists(video_path):
            missing_videos += 1
            continue

        if isinstance(captions, str):
            captions = [captions]

        for caption in captions:
            caption = str(caption).strip()
            if caption:
                pairs.append(
                    {
                        "video_id": item["video_id"],
                        "video_path": video_path,
                        "caption": caption,
                    }
                )

    if len(pairs) == 0:
        raise RuntimeError(f"No training pairs found in {json_path}. Check video_root and file names.")

    print(f"Loaded {len(records)} video records from {json_path}")
    print(f"Created {len(pairs)} text-video pairs")
    print(f"Skipped {missing_videos} records because the video file was not found under {video_root}")
    return pairs

In [18]:
# ============================================================
# BLOCK 4: Video frame loading and transforms
# ============================================================

class VideoFrameReader:
    def __init__(self, image_size: int, num_frames: int):
        self.image_size = image_size
        self.num_frames = num_frames
        self.transform = transforms.Compose(
            [
                transforms.Resize((image_size, image_size), interpolation=transforms.InterpolationMode.BICUBIC),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=(0.48145466, 0.4578275, 0.40821073),
                    std=(0.26862954, 0.26130258, 0.27577711),
                ),
            ]
        )

    def _sample_indices(self, total_frames: int) -> List[int]:
        if total_frames <= 0:
            return [0] * self.num_frames
        if total_frames < self.num_frames:
            idx = np.linspace(0, total_frames - 1, self.num_frames)
            return [int(round(x)) for x in idx]
        idx = np.linspace(0, total_frames - 1, self.num_frames)
        return [int(x) for x in idx]

    def read_video(self, video_path: str) -> torch.Tensor:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise RuntimeError(f"Could not open video: {video_path}")

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_indices = self._sample_indices(total_frames)

        frames = []
        for idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ok, frame = cap.read()
            if not ok or frame is None:
                if frames:
                    frame_tensor = frames[-1].clone()
                    frames.append(frame_tensor)
                    continue
                else:
                    cap.release()
                    raise RuntimeError(f"Failed to read frame {idx} from {video_path}")
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = Image.fromarray(frame)
            frame_tensor = self.transform(image)
            frames.append(frame_tensor)

        cap.release()
        return torch.stack(frames, dim=0)

In [19]:
# ============================================================
# BLOCK 5: Dataset and collate function
# ============================================================

class MSVDPairDataset(Dataset):
    def __init__(self, pairs: List[Dict[str, str]], frame_reader: VideoFrameReader):
        self.pairs = pairs
        self.frame_reader = frame_reader

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> Dict[str, Any]:
        item = self.pairs[index]
        frames = self.frame_reader.read_video(item["video_path"])
        return {
            "video_id": item["video_id"],
            "video_path": item["video_path"],
            "caption": item["caption"],
            "frames": frames,
        }


@dataclass
class Batch:
    pixel_values: torch.Tensor
    captions: List[str]
    video_ids: List[str]
    video_paths: List[str]


class Collator:
    def __call__(self, batch: List[Dict[str, Any]]) -> Batch:
        pixel_values = torch.stack([x["frames"] for x in batch], dim=0)
        captions = [x["caption"] for x in batch]
        video_ids = [x["video_id"] for x in batch]
        video_paths = [x["video_path"] for x in batch]
        return Batch(
            pixel_values=pixel_values,
            captions=captions,
            video_ids=video_ids,
            video_paths=video_paths,
        )

In [20]:
# ============================================================
# BLOCK 6: CLIP video-text model wrapper
# ============================================================

class VideoTextCLIP(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(model_name)

    def encode_video(self, pixel_values: torch.Tensor) -> torch.Tensor:
        bsz, num_frames, c, h, w = pixel_values.shape
        flat = pixel_values.view(bsz * num_frames, c, h, w)
        image_features = self.clip.get_image_features(pixel_values=flat)
        image_features = image_features.view(bsz, num_frames, -1)
        video_features = image_features.mean(dim=1)
        video_features = F.normalize(video_features, dim=-1)
        return video_features

    def encode_text(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        text_features = self.clip.get_text_features(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        text_features = F.normalize(text_features, dim=-1)
        return text_features

    def forward(
        self,
        pixel_values: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        video_features = self.encode_video(pixel_values)
        text_features = self.encode_text(input_ids, attention_mask)
        logit_scale = self.clip.logit_scale.exp().clamp(max=100.0)
        return video_features, text_features, logit_scale

In [21]:
# ============================================================
# BLOCK 7: Fine-tuning control for CLIP layers
# ============================================================

def freeze_module(module: nn.Module) -> None:
    for p in module.parameters():
        p.requires_grad = False


def unfreeze_module(module: nn.Module) -> None:
    for p in module.parameters():
        p.requires_grad = True


def apply_finetune_policy(model: VideoTextCLIP, args: argparse.Namespace) -> None:
    freeze_module(model)

    # Always train temperature unless you want to lock it manually.
    model.clip.logit_scale.requires_grad = True

    if args.train_text_projection_only:
        unfreeze_module(model.clip.text_projection)
        return

    if args.train_vision_projection_only:
        unfreeze_module(model.clip.visual_projection)
        return

    # Vision side
    if not args.freeze_vision:
        unfreeze_module(model.clip.vision_model.post_layernorm)
        unfreeze_module(model.clip.visual_projection)
        vision_layers = model.clip.vision_model.encoder.layers
        n = min(args.unfreeze_last_n_vision, len(vision_layers))
        for layer in vision_layers[-n:]:
            unfreeze_module(layer)

    # Text side
    if not args.freeze_text:
        unfreeze_module(model.clip.text_model.final_layer_norm)
        unfreeze_module(model.clip.text_projection)
        text_layers = model.clip.text_model.encoder.layers
        n = min(args.unfreeze_last_n_text, len(text_layers))
        for layer in text_layers[-n:]:
            unfreeze_module(layer)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable:,} / {total:,}")

In [22]:
# ============================================================
# BLOCK 8: Loss, optimizer, scheduler
# ============================================================

def contrastive_loss(video_features: torch.Tensor, text_features: torch.Tensor, logit_scale: torch.Tensor) -> torch.Tensor:
    logits_per_video = logit_scale * video_features @ text_features.t()
    logits_per_text = logits_per_video.t()
    targets = torch.arange(video_features.size(0), device=video_features.device)
    loss_v2t = F.cross_entropy(logits_per_video, targets)
    loss_t2v = F.cross_entropy(logits_per_text, targets)
    return 0.5 * (loss_v2t + loss_t2v)


def build_optimizer(model: nn.Module, lr: float, weight_decay: float) -> torch.optim.Optimizer:
    decay_params = []
    no_decay_params = []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim < 2 or name.endswith("bias") or "layer_norm" in name.lower() or "norm" in name.lower():
            no_decay_params.append(p)
        else:
            decay_params.append(p)

    optimizer = torch.optim.AdamW(
        [
            {"params": decay_params, "weight_decay": weight_decay},
            {"params": no_decay_params, "weight_decay": 0.0},
        ],
        lr=lr,
        betas=(0.9, 0.98),
        eps=1e-6,
    )
    return optimizer

In [23]:
# ============================================================
# BLOCK 9: Training and validation loops
# ============================================================

def tokenize_text(tokenizer: CLIPTokenizerFast, captions: List[str], max_text_len: int, device: torch.device) -> Dict[str, torch.Tensor]:
    encoded = tokenizer(
        captions,
        padding=True,
        truncation=True,
        max_length=max_text_len,
        return_tensors="pt",
    )
    return {k: v.to(device) for k, v in encoded.items()}


def run_train_epoch(
    model: VideoTextCLIP,
    tokenizer: CLIPTokenizerFast,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler,
    scaler: torch.cuda.amp.GradScaler,
    device: torch.device,
    epoch: int,
    args: argparse.Namespace,
) -> float:
    model.train()
    running_loss = 0.0
    step_count = 0
    optimizer.zero_grad(set_to_none=True)
    start_time = time.time()

    for step, batch in enumerate(loader):
        pixel_values = batch.pixel_values.to(device, non_blocking=True)
        text_inputs = tokenize_text(tokenizer, batch.captions, args.max_text_len, device)

        use_amp = args.amp and device.type == "cuda"
        with torch.cuda.amp.autocast(enabled=use_amp):
            video_features, text_features, logit_scale = model(
                pixel_values=pixel_values,
                input_ids=text_inputs["input_ids"],
                attention_mask=text_inputs["attention_mask"],
            )
            loss = contrastive_loss(video_features, text_features, logit_scale)
            loss = loss / args.grad_accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % args.grad_accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        running_loss += loss.item() * args.grad_accum_steps
        step_count += 1

        if (step + 1) % args.log_every == 0:
            elapsed = time.time() - start_time
            avg_loss = running_loss / step_count
            print(
                f"[Train] Epoch {epoch} Step {step + 1}/{len(loader)} "
                f"Loss {avg_loss:.4f} "
                f"LR {scheduler.get_last_lr()[0]:.7f} "
                f"Elapsed {format_seconds(elapsed)}"
            )

    return running_loss / max(step_count, 1)


@torch.no_grad()
def run_val_epoch(
    model: VideoTextCLIP,
    tokenizer: CLIPTokenizerFast,
    loader: DataLoader,
    device: torch.device,
    args: argparse.Namespace,
    epoch: int,
) -> float:
    model.eval()
    running_loss = 0.0
    step_count = 0

    for step, batch in enumerate(loader):
        pixel_values = batch.pixel_values.to(device, non_blocking=True)
        text_inputs = tokenize_text(tokenizer, batch.captions, args.max_text_len, device)

        use_amp = args.amp and device.type == "cuda"
        with torch.cuda.amp.autocast(enabled=use_amp):
            video_features, text_features, logit_scale = model(
                pixel_values=pixel_values,
                input_ids=text_inputs["input_ids"],
                attention_mask=text_inputs["attention_mask"],
            )
            loss = contrastive_loss(video_features, text_features, logit_scale)

        running_loss += loss.item()
        step_count += 1

    avg_loss = running_loss / max(step_count, 1)
    print(f"[Val] Epoch {epoch} Loss {avg_loss:.4f}")
    return avg_loss

In [24]:
# ============================================================
# BLOCK 10: Checkpoint save and resume
# ============================================================

def save_checkpoint(
    path: str,
    model: VideoTextCLIP,
    optimizer: torch.optim.Optimizer,
    scheduler,
    scaler: torch.cuda.amp.GradScaler,
    epoch: int,
    best_val_loss: float,
    args: argparse.Namespace,
) -> None:
    state = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "best_val_loss": best_val_loss,
        "args": vars(args),
    }
    torch.save(state, path)
    print(f"Saved checkpoint to {path}")


def maybe_resume(
    resume_path: str,
    model: VideoTextCLIP,
    optimizer: torch.optim.Optimizer,
    scheduler,
    scaler: torch.cuda.amp.GradScaler,
    device: torch.device,
) -> Tuple[int, float]:
    if not resume_path:
        return 1, float("inf")

    ckpt = torch.load(resume_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    start_epoch = int(ckpt["epoch"]) + 1
    best_val_loss = float(ckpt.get("best_val_loss", float("inf")))
    print(f"Resumed from {resume_path} at epoch {start_epoch}")
    return start_epoch, best_val_loss

In [25]:
# ============================================================
# BLOCK 11: Main training pipeline
# ============================================================

def main() -> None:
    args = parse_args()
    ensure_dir(args.output_dir)
    set_seed(args.seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_pairs = build_pairs_from_msvd_json(args.train_json, args.video_root)
    val_pairs = build_pairs_from_msvd_json(args.val_json, args.video_root)

    frame_reader = VideoFrameReader(image_size=args.image_size, num_frames=args.num_frames)
    train_dataset = MSVDPairDataset(train_pairs, frame_reader)
    val_dataset = MSVDPairDataset(val_pairs, frame_reader)

    collator = Collator()
    train_loader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers,
        pin_memory=True,
        drop_last=True,
        collate_fn=collator,
        persistent_workers=args.num_workers > 0,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=True,
        drop_last=False,
        collate_fn=collator,
        persistent_workers=args.num_workers > 0,
    )

    tokenizer = CLIPTokenizerFast.from_pretrained(args.model_name)
    model = VideoTextCLIP(args.model_name)
    apply_finetune_policy(model, args)
    model.to(device)

    optimizer = build_optimizer(model, lr=args.lr, weight_decay=args.weight_decay)
    total_update_steps = math.ceil(len(train_loader) / args.grad_accum_steps) * args.epochs
    warmup_steps = int(total_update_steps * args.warmup_ratio)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_update_steps,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=args.amp and device.type == "cuda")

    start_epoch, best_val_loss = maybe_resume(
        args.resume,
        model,
        optimizer,
        scheduler,
        scaler,
        device,
    )

    save_json(vars(args), os.path.join(args.output_dir, "train_args.json"))

    history = []
    train_start = time.time()

    for epoch in range(start_epoch, args.epochs + 1):
        epoch_start = time.time()

        train_loss = run_train_epoch(
            model=model,
            tokenizer=tokenizer,
            loader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            device=device,
            epoch=epoch,
            args=args,
        )

        val_loss = run_val_epoch(
            model=model,
            tokenizer=tokenizer,
            loader=val_loader,
            device=device,
            args=args,
            epoch=epoch,
        )

        epoch_time = time.time() - epoch_start
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "epoch_time_sec": epoch_time,
            }
        )
        save_json(history, os.path.join(args.output_dir, "history.json"))

        print(
            f"Epoch {epoch} done | Train Loss {train_loss:.4f} | "
            f"Val Loss {val_loss:.4f} | Time {format_seconds(epoch_time)}"
        )

        if epoch % args.save_every == 0:
            ckpt_path = os.path.join(args.output_dir, f"checkpoint_epoch_{epoch}.pt")
            save_checkpoint(ckpt_path, model, optimizer, scheduler, scaler, epoch, best_val_loss, args)

        last_path = os.path.join(args.output_dir, "last.pt")
        save_checkpoint(last_path, model, optimizer, scheduler, scaler, epoch, best_val_loss, args)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(args.output_dir, "best.pt")
            save_checkpoint(best_path, model, optimizer, scheduler, scaler, epoch, best_val_loss, args)
            print(f"New best checkpoint at epoch {epoch} with val loss {val_loss:.4f}")

    total_time = time.time() - train_start
    print(f"Training finished in {format_seconds(total_time)}")
    print(f"Best val loss: {best_val_loss:.4f}")

In [26]:
# ============================================================
# BLOCK 12: Entry point
# ============================================================

if __name__ == "__main__":
    main()

usage: colab_kernel_launcher.py [-h] --train_json TRAIN_JSON --val_json
                                VAL_JSON --video_root VIDEO_ROOT --output_dir
                                OUTPUT_DIR [--model_name MODEL_NAME]
                                [--image_size IMAGE_SIZE]
                                [--max_text_len MAX_TEXT_LEN]
                                [--num_frames NUM_FRAMES]
                                [--batch_size BATCH_SIZE]
                                [--num_workers NUM_WORKERS] [--epochs EPOCHS]
                                [--lr LR] [--weight_decay WEIGHT_DECAY]
                                [--warmup_ratio WARMUP_RATIO]
                                [--grad_accum_steps GRAD_ACCUM_STEPS]
                                [--max_grad_norm MAX_GRAD_NORM]
                                [--freeze_vision] [--freeze_text]
                                [--train_text_projection_only]
                                [--train_vision_projection_only]
     

Traceback (most recent call last):
  File "/usr/lib/python3.12/argparse.py", line 1943, in _parse_known_args2
    namespace, args = self._parse_known_args(args, namespace, intermixed)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 2230, in _parse_known_args
    raise ArgumentError(None, _('the following arguments are required: %s') %
argparse.ArgumentError: the following arguments are required: --train_json, --val_json, --video_root, --output_dir

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_508/3372256590.py", line 6, in <cell line: 0>
    main()
  File "/tmp/ipykernel_508/2762124506.py", line 6, in main
    args = parse_args()
           ^^^^^^^^^^^^
  File "/tmp/ipykernel_

TypeError: object of type 'NoneType' has no len()